# Tutorial passo a passo da Análise exploratória dos dados

No geral as ferramentas utilizadas para construir uma análise exploratória são: pandas, seaborn, matplotlib, folium, plotly, numpy, statsmodels e o label encoder

Vamos instalar a versão do matplotlib 3.5.2 ou superior por conter alguns recursos que falicitam na visualização dos dados.

In [ ]:
# instalações
!pip install matplotlib==3.5.2

As importações podem ocorrer em qualquer parte do código, mas como uma boa prática é sempre recomendado utilizar no topo do script. Segue abaixo o exemplo para realizar as importações

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
import plotly.figure_factory as ff
from scipy.cluster import hierarchy
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder

Na linha abaixo é utilizado o comando read_csv do pandas para carregar a base de dados

In [ ]:
base = pd.read_csv('base_sem_identificacao.csv', sep=';')

Para ter uma visão geral de todas as colunas pode-se utilizar o comando abaixo

In [ ]:
base.columns

# Análise univariada

## Plotando gráficos

Existem várias ferramentas formas de plotar os gráficos, neste caso, vamos utilizar o seaborn.

In [ ]:
# Nesta linha estou definindo o tamanho
sns.set(rc={'figure.figsize':(11.7,8.27)})

# Nessa linha estou defindo qual valor será apresentado no eixo Y e no X ordenando de forma decrescente 
ax = sns.countplot(y = base.iloc[:,0],
              data = base,
              order = base.iloc[:,0].value_counts()[:10].index)

# Aqui estou configurando os valores que ficaram no topo do gráfico
width = 0.35
valores = base.iloc[:,0].value_counts()[:10].values
ax.bar_label(container=ax.containers[0], labels=valores)

# Título para o eixo Y
ax.set_ylabel('')

# Título para o eixo X
ax.set_xlabel('Alunos')

# Título para o gráfico
ax.set_title(base.iloc[:,0].name)


## Salvando figura
fig = ax.get_figure()
fig.savefig('1.1 - concludente e evasores - Curso CeT.png',bbox_inches='tight')

# Exibindo o gráfico na tela
plt.show()

In [ ]:
# contando a frequência das categorias em valores absolutos
situacaovalores = base.iloc[:,0].value_counts()

# contantando a frequência das categorias em valores percentuais
situacaopercentual = base.iloc[:,0].value_counts(normalize=True).round(2) * 100

In [ ]:
# criando uma tabela para apresentar os valores absolutos e percentuais
dfsistuacao = pd.DataFrame({'Contagem': situacaovalores, 'Frequência':situacaopercentual})
dfsistuacao.reset_index(inplace=True )
dfsistuacao.columns = ['Situação', 'Contagem', 'Frequência (%)']
dfsistuacao.style.set_caption("Alunos - CeT")

,Situação,Contagem,Frequência (%)
0,Concludente/Egresso,184,80.000000
1,Evadido/Desistente,45,20.000000


Existem casos que é necessário utilizar o comando replace que é para substituir valores deixando o conjunto no mesmo padrão dos dados

In [ ]:
# replace dos dados fora do padrão primeiro campo é o valor na base e o segundo é o valor que será substituído
base.iloc[:,2] = base.iloc[:,2].replace('2013', '2013.1')
base.iloc[:,2] = base.iloc[:,2].replace('2021-1', '2021.1')
base.iloc[:,2] = base.iloc[:,2].replace('2011.1 não tenho certeza', 'NAO LEMBRO ')
base.iloc[:,2] = base.iloc[:,2].replace('2020.1 1 semestre ', '2020.1')

# TABELAS DE CONTIGÊNCIA

*As tabelas de contingência são usadas para registrar observações categóricas de duas ou mais variáveis. No uso desse tipo de tabela é comum se pretender investigar se as variáveis estudadas têm alguma associação com a variável alvo, neste caso a nossa variável alvo é a conclusão ou evasão do aluno.*

In [ ]:
# criando uma função para conversão de valores percentuais
def percConvert(ser):
  return ser/float(ser[-1])

In [ ]:
# selecionando valores
base2 = base.iloc[:,0:32]

In [ ]:
# Valores absolutos

In [ ]:
# criando estrutura de repetição para acessar todas as colunas e gerar a tabela de contigência
for i in range(1,len(base2.columns)):
    ax = sns.heatmap(pd.crosstab(base2.iloc[:,i],base2.iloc[:,0],margins=True),cmap="YlGnBu", annot=True, cbar=False, fmt="d")
    plt.show(ax)
    print('------------------------------------------------------------------------------------')

In [ ]:
# Valores relativos
for i in range(1,len(base2.columns)):
    ax = sns.heatmap(pd.crosstab(base2.iloc[:,i],base2.iloc[:,0],margins=True).apply(percConvert, axis=1),cmap="YlGnBu", annot=True, cbar=False)
    plt.show(ax)
    print('------------------------------------------------------------------------------------')

# Correlação de Pearson

In [ ]:
basespss = pd.read_csv('spss2.csv', sep=';')
basespss = basespss.iloc[:,0:32]
base2 = basespss

In [ ]:
# aplicando o label encoder é para converter as categorias em números
encoder = LabelEncoder()
for i in range(0,len(base2.columns)):
    base2[base2.columns[i]] = encoder.fit_transform(base2[base2.columns[i]])

In [ ]:
# Criando variável para armazenar os valores das correlações já calculadas
corr = base2.corr()

In [ ]:
# Criando mapa de calor para representar a intensidade das correlações
sns.set(font_scale=4)
f, ax = plt.subplots(figsize=(80, 60))
ax = sns.heatmap(corr,
            annot = True,
            fmt = '.2f',
            cmap='Blues')
plt.title('Correlação entre variáveis do dataset')
plt.show()

## Selecionando as melhores caracteríricas

In [ ]:
# utilizando o algoritmo random forest para selecionar as melhores características
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X = base2.iloc[:,1:33]
y = base2.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)# Treinando modelo
model  = RandomForestClassifier()
model.fit(X_train, y_train)
model.feature_importances_

In [ ]:
# gerando gráfico ordenando o nível de importância do maior para o menor
sns.set(font_scale=5)
f, ax = plt.subplots(figsize=(60, 40))
importances = pd.Series(data=model.feature_importances_, index=base2.iloc[:,1:33].columns)
sns.barplot(x=importances.sort_values(ascending=False), y=importances.sort_values(ascending=False).index, orient='h').set_title('Importância de cada feature')

In [ ]:
# salvando o conjunto dos selecionados em um data frame
selecionados.to_csv('selecionados.csv', index=False, sep=';')